# 01 - Data Profiling

Column-level statistics, semantic type inference, null rate visualization, and table-level profiling of synthetic corrections data.

**Toolkit modules used**: `cj_data_quality.profiling`, `cj_data_quality.visualization`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from cj_data_quality.notebook_utils import setup_notebook, display_table_profile, style_null_rates
from cj_data_quality.sample_data import generate_and_load
from cj_data_quality.profiling.column_profiler import profile_column
from cj_data_quality.profiling.table_profiler import profile_table
from cj_data_quality.profiling.type_inference import infer_column_type
from cj_data_quality.visualization.plots import plot_null_rate_bars, plot_profile_summary

setup_notebook()
print("Setup complete.")

## Load Sample Data

In [ ]:
df = generate_and_load(n_records=50000, seed=42)
print(f"Shape: {df.shape}")
print(f"States: {df['state_code'].nunique()}")
df.head()

## Table Profile

Profile the entire table: null rates, cardinality, duplicates, and column-level statistics.

In [ ]:
table_profile = profile_table(df, table_name="corrections_data")
print(f"Rows: {table_profile.row_count:,}")
print(f"Columns: {table_profile.column_count}")
print(f"Duplicates: {table_profile.duplicate_row_count:,} ({table_profile.duplicate_rate:.1%})")
print(f"Overall Null Rate: {table_profile.overall_null_rate:.1%}")

In [ ]:
profile_df = display_table_profile(table_profile)
profile_df

## Null Rate Visualization

Color-coded by severity: green (< 5%), amber (5-20%), red (> 50%).

In [ ]:
fig = plot_null_rate_bars(table_profile.column_profiles, title="Corrections Data: Column Null Rates")
plt.show()

## Profile Summary Dashboard

In [ ]:
fig = plot_profile_summary(table_profile)
plt.show()

## Semantic Type Inference

The type inference module uses a decision tree to classify columns as identifiers, temporal, numeric, categorical, boolean, free text, or unknown.

In [ ]:
type_results = []
for col in df.columns:
    inferred = infer_column_type(df[col], col)
    type_results.append({"Column": col, "Inferred Type": inferred.value})

pd.DataFrame(type_results)

## Deep Dive: Numeric Columns

Examine distribution statistics for numeric fields.

In [ ]:
numeric_profiles = [p for p in table_profile.column_profiles if p.numeric_stats is not None]
for p in numeric_profiles:
    ns = p.numeric_stats
    print(f"\n--- {p.column_name} ---")
    print(f"  Mean: {ns.mean:.1f}, Median: {ns.median:.1f}, Std: {ns.std:.1f}")
    print(f"  Range: [{ns.min_value:.0f}, {ns.max_value:.0f}]")
    print(f"  IQR: [{ns.p25:.0f}, {ns.p75:.0f}]")
    if ns.skewness is not None:
        print(f"  Skewness: {ns.skewness:.3f}")

## Per-State Null Rate Analysis

Some states report data more completely than others.

In [ ]:
state_null_rates = df.groupby("state_code").apply(
    lambda g: g.isnull().mean().mean()
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
state_null_rates.plot.barh(ax=ax, color="#004C6D")
ax.set_xlabel("Mean Null Rate")
ax.set_title("Top 15 States by Overall Null Rate", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Key Findings

1. **Demographic fields** (race, ethnicity) have the highest null rates, varying significantly by state
2. **Date fields** show moderate missingness, with some states having 30%+ null admission dates
3. **Identifier fields** (person_id, state_code) are complete across the board
4. **Population metrics** are generally well-populated

These patterns mirror real-world CJ data challenges documented in Recidiviz's blog posts.